In [12]:
# import modal 
from torch import nn
import torch 
import os
import pandas as pd
from dataset import ESC50Dataset
import torchvision
import numpy as np
# import sys
np.random.seed(42)
class AudioClassifier(nn.Module):
    def __init__(self, num_classes=50):
        super(AudioClassifier, self).__init__() 
        self.num_classes = num_classes

        self.conv1 = nn.Conv2d(1 ,32 , kernel_size=(3,3) , stride=(1,1),padding=(1,1))
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=(2,2), stride=(2,2))
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3,3), stride=(1,1), padding=(1,1))
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=(2,2), stride=(2,2)) 

        self.conv3 = nn.Conv2d(64, 128, kernel_size=(3,3), stride=(1,1), padding=(1,1))
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(kernel_size=(2,2), stride=(2,2)) 

        self.conv4 = nn.Conv2d(128, 256, kernel_size=(3,3), stride=(1,1), padding=(1,1))
        self.bn4 = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d(kernel_size=(2,2), stride=(2,2)) 
        self.global_pool = nn.AdaptiveAvgPool2d((1,1))
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256, num_classes)

        
        
    def forward(self,x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool3(torch.relu(self.bn3(self.conv3(x))))
        x = self.pool4(torch.relu(self.bn4(self.conv4(x))))
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

    
data_dir = os.path.join("data","ESC-50-master")
audio_dir = os.path.join(data_dir, "audio")
table_dir = os.path.join(data_dir, "meta")

df = pd.read_csv(os.path.join(table_dir, "esc50.csv"))
df.drop(columns=["src_file","take","esc10"], inplace=True)

folds = df["fold"].unique()
val_fold = np.random.choice(folds)
train_df = df[df["fold"] != val_fold]
val_df = df[df["fold"] == val_fold]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = torchvision.transforms.Compose([
        # torchvision.transforms.Resize((128,128)), #resize should never be applied to audio data as it can distort the frequency information, 
        #the incoming mel spectrogram is already 128 by 128 (made sure by choosing hop_length=625) but the best thing to do instead of resizing is center cropping
        torchvision.transforms.CenterCrop((128,128)),
        # torchaudio.transforms.FrequencyMasking(freq_mask_param=8),
        # torchaudio.transforms.TimeMasking(time_mask_param=12)
    ])

    
val_dataset = ESC50Dataset(val_df, audio_dir,transform=None)
model = AudioClassifier(num_classes=50)
chkpt = torch.load("best_model.pth",map_location=device)
model.load_state_dict(chkpt["model_state_dict"])




<All keys matched successfully>

In [13]:
data_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

In [ ]:
with torch.no_grad():
    model.eval()
    correct_count=0
    for batch in data_loader:
        inputs, labels = batch
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs) 
        predicted_labels = torch.argmax(outputs, dim=1)
        correct_count += (predicted_labels ==labels).sum().item()
    print(f"Validation Accuracy: {correct_count/len(val_dataset)*100:.2f}%")

Validation Accuracy: 82.00%
